<a href="https://colab.research.google.com/github/m1raynee/term4-alg/blob/main/lab1/lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# __Последнее задание выдано, отчет можно прикреплять.__

## __Задание 1__

В задании часто будет встречаться слово «символ». Подразумевается объект ассоциированный с конкретным кодом, а не текстовый символ, если кодировка не указана явно. Например, цветное изображение представляется последовательностью символов с трехбайтным кодом.

#### 1. Тестовые данные
Подготовить следующий тестовый набор:

- enwik7 (первые 1E+7 байт enwik9 [https://mattmahoney.net/dc/textdata.html])
- текст на русском языке, объемом не менее 200Кб
- бинарный файл объемом не менее 1 Мб, содержимое которого не представляет собой осмысленный текст. Например, произвольный .exe файл
- Черно-белое изображение, размером не менее 800x600 пикселей
- Изображение в оттенках серого, размером не менее 800x600 пикселей
- Цветное изображение, размером не менее 800x600 пикселей

Перевести изображения в собственный raw формат, в котором один/три байта задают значение цветности пикселя. Дополнить файл метаданными о типе (чб/оттенки серого/цветное). Сравнить размер получившегося файла с размером изображения и сделать приблизительную оценку коэффициента сжатия исходного изображения в формате jpg, png и т.д.

#### 2. Run-length encoding
- Написать функцию кодирования и декодирования алгоритма RLE, где входными и выходными данными служит байтовая строка, длина кода символов - 8 бит, длина кода управляющего символа, задающего количество повторов – 8 бит.

__Пример__:
0xCF 0xCF 0xCF 0xCF 0xCF -> 0x05 0xCF

- Добавить в функции кодирования и декодирования обработку неповторяющихся последовательностей с помощью старшего бита в управляющем символе. Подумайте о том, последовательности какой длины вы могли записать раньше и сколько их будет сейчас.

__Пример__:

0xCF 0xCF 0xCF 0xCF 0xCF -> 0x05 0xCF

0xCF 0xCE 0xCF 0xCE 0xCF -> 0x85 0xCF 0xCE 0xCF 0xCE 0xCF [0x85 = 0b10000101]

0xCF 0xCE 0xCF 0xCE 0xCF 0xCF 0xCF 0xCF 0xCF 0xCF -> 0x84 0xCF 0xCE 0xCF 0xCE 0x05 0xCF

- Добавить в функции кодирования и декодирования аргументы Ms и Mc: длину кода символа в байтах и длину управляющего символа в байтах

__Пример__:

Ms = 2:

0xCF 0xCE 0xCF 0xCE 0xCF 0xCE -> 0x03 0xCF 0xCE

Ms = 3:

0xCF 0xCE 0xCF 0xCF 0xCE 0xCF -> 0x02 0xCF 0xCE 0xCF

- Подумайте насколько применима данная функция для сжатия изображений в raw формате. Какие значения Ms и Mc стоит выбрать для черно-белых и gray-scale изображений, а какое для цветных?

- Подумайте какая проблема возникает при применении данной функции к байтовой последовательности представляющей собой последовательность кодов символов в кодировке UTF-8.  Попробуйте решить эту проблему.

- Реализуйте чтение и запись битовых строк из файла. Добавьте метаданные в файл с закодированным сообщением о длине кода символа и длине кода управляющего символа.
Убедитесь, что последовательность: чтение-> кодирование RLE -> запись в файл закодированного сообщения -> чтение закодированного сообщения из файла -> декодирование RLE -> запись в файл декодированного сообщения возвращает вам исходный файл

- Примените написанные функции к простейшим тестовым данным и убедитесь, что они работают корректно для всех случаев (байтовая строка с различной длиной равномерного кода символов, текст с кириллицей в кодировке UTF-8).

- Сделайте оценку коэффициента сжатия вашей реализацией RLE для всех тестовых файлов путем анализа файла. Примените получившийся компрессор ко всем тестовым файлам, рассчитайте коэффициент сжатия и сравните его с полученной выше оценкой.

In [1]:
import os
import struct
from PIL import Image


class ImageProcessor:
    """Конвертация изображений в кастомный RAW формат."""

    @staticmethod
    def to_raw(image_path: str, output_path: str, **convert_kwargs):
        with Image.open(image_path) as old_img:

            img = old_img
            if convert_kwargs:
                img = old_img.convert(**convert_kwargs)

            mode = img.mode
            width, height = img.size
            mode_id = {'1': 0, 'L': 1, 'RGB': 2}.get(mode, 2)

            raw_data = img.tobytes()

            # [1 байт тип][4 байт ширина][4 байт высота][данные]
            # 0 - BW, 1 - Grayscale, 2 - RGB
            with open(output_path, 'wb') as f:
                f.write(struct.pack('BII', mode_id, width, height))
                f.write(raw_data)

            original_size = os.path.getsize(image_path)
            raw_size = os.path.getsize(output_path)
            ratio = original_size / raw_size if raw_size > 0 else 0
            return mode_id, raw_size, ratio

In [2]:
class RLECompressor:
    """
    RLE компрессор
    """

    @staticmethod
    def encode(data: bytes, ms: int = 1, mc: int = 1) -> bytes:
        if not data:
            return b""

        result = bytearray()
        i = 0
        n = len(data)

        # mc=1 => max = 2^7 - 1 = 127
        max_run = (1 << (8 * mc - 1)) - 1

        while i < n:
            symbol = data[i : i + ms]
            run_len = 1

            while (
                i + run_len * ms < n
                # срез по количеству байт
                and data[i + run_len * ms : i + (run_len + 1) * ms] == symbol
                and run_len < max_run
            ):
                run_len += 1

            if run_len > 1:
                # повторяющийся блок (старший бит 0)
                result.extend(run_len.to_bytes(mc))
                result.extend(symbol)
                i += run_len * ms
            else:
                # литеральный блок (старший бит 1)
                curr_i = i + ms
                count = 1

                while curr_i < n and count < max_run:
                    # если начинаются повторы, переключение на RLE
                    if curr_i + ms < n and data[curr_i : curr_i + ms] == data[curr_i + ms : curr_i + 2 * ms]:
                        break
                    curr_i += ms
                    count += 1

                control = count | (1 << (8 * mc - 1))
                result.extend(control.to_bytes(mc))
                result.extend(data[i : curr_i])
                i = curr_i

        return bytes(result)

    @staticmethod
    def decode(data: bytes, ms: int = 1, mc: int = 1) -> bytes:
        if not data:
            return b""

        result = bytearray()
        i = 0
        n = len(data)
        # для mc = 1: 10000000 и 01111111 соответственно
        flag_mask = 1 << (8 * mc - 1)
        value_mask = flag_mask - 1

        while i < n:
            header_bytes = data[i : i + mc]
            control = int.from_bytes(header_bytes)
            i += mc

            count = control & value_mask

            if control & flag_mask:
                for _ in range(count):
                    result.extend(data[i : i + ms])
                    i += ms
            else:
                symbol = data[i : i + ms]
                i += ms
                for _ in range(count):
                    result.extend(symbol)

        return bytes(result)

    @staticmethod
    def save_compressed(filepath: str, data: bytes, ms: int, mc: int) -> None:
        # [1 байт Ms][1 байт Mc][данные]
        with open(filepath, 'wb') as f:
            f.write(struct.pack('BB', ms, mc))
            f.write(data)

    @staticmethod
    def load_compressed(filepath: str) -> tuple[bytes, int, int]:
        with open(filepath, 'rb') as f:
            ms, mc = struct.unpack('BB', f.read(2))
            data = f.read()
            return data, ms, mc


rle = RLECompressor

In [3]:
def test_rle_logic():
    """Тестирование примеров из задания."""

    print("--- Примеры RLE ---")

    # Пример 1: 0xCF * 5 -> 0x05 0xCF (при Mc=1, Ms=1)
    data1 = b'\xCF' * 5
    enc1 = RLECompressor.encode(data1)
    print(f"Повторы: {data1.hex()} -> {enc1.hex()}")

    # Пример 2: Без повторов
    data2 = bytes([0xCF, 0xCE, 0xCF, 0xCE, 0xCF])
    enc2 = rle.encode(data2)
    # 0x85 + данные
    print(f"Без повторов: {data2.hex()} -> {enc2.hex()}")

    data3 = bytes([0xCF, 0xCE, 0xCF, 0xCE, 0xCF, 0xCE])
    enc3 = rle.encode(data3, ms=2, mc=1)
    print(f"ms=2: {data3.hex()} -> {enc3.hex()}")

    data4 = bytes([0xCF, 0xCE, 0xCF, 0xCF, 0xCE, 0xCF])
    enc4 = rle.encode(data4, ms=3, mc=1)
    print(f"ms=3: {data4.hex()} -> {enc4.hex()}")

def test_rle_cycle_integrity(data: bytes = None, *, omit_prints: bool = False, keep_file: bool = False) -> bool:
    """Тестирование цикла кодирования"""
    print("--- Цикл кодирования RLE ---")

    if data is None:
        data = bytes([0xCF, 0xCE, 0xCF, 0xCE, 0xCF, 0xCE])

    msc = (2, 1)

    if not omit_prints:
        print(f"data:    {data.hex()}")
        print(f"{msc = }")

    rle.save_compressed("resources/test_cycle", rle.encode(data, *msc), *msc)
    new_data, *new_msc = rle.load_compressed("resources/test_cycle")

    if not keep_file:
        os.remove("resources/test_cycle")

    if not omit_prints:
        print(f"compressed data: {new_data.hex()}")
        print(f"{new_msc = }")

    decoded_data = rle.decode(new_data, *new_msc)

    integrity = data == decoded_data
    if not omit_prints:
        print(f"decoded: {data.hex()}")
        print(f"{integrity = }")

    return integrity

In [4]:
test_rle_logic()
test_rle_cycle_integrity()

--- Примеры RLE ---
Повторы: cfcfcfcfcf -> 05cf
Без повторов: cfcecfcecf -> 85cfcecfcecf
ms=2: cfcecfcecfce -> 03cfce
ms=3: cfcecfcfcecf -> 02cfcecf
--- Цикл кодирования RLE ---
data:    cfcecfcecfce
msc = (2, 1)
compressed data: 03cfce
new_msc = [2, 1]
decoded: cfcecfcecfce
integrity = True


True

## __Задание 2__

####  1. Энтропийное кодирование
* Написать функцию расчета энтропии сообщения в зависимости от длины кода символа.\
 __Входные данные:__ байтовая строка и длина кода символов в байта(равномерное кодирование) \
__Выходные данные:__ значение энтропии. \
Взять любой текст на английском, отфильтровать из него текстовые символы, имеющие юникод выше 127, и исследовать зависимость значения энтропии всего сообщения от длины кода символов (1-4 байт), выведя её на график. Как можно интерпретировать эти результаты?
* Написать функции прямого и обратного преобразования MTF. \
__Входные и выходные данные:__ байтовая строка.\
Как стоило бы модифицировать ваши функции, если бы вы работали с символами в кодировке UTF-8? Исследуйте влияние MTF на значение энтропии текста из тестовой выборки и сделайте вывод о применимости алгоритма перед энтропийным кодированием.
*   Написать функции, реализующую кодирование (сжатие) и декодирование с помощью алгоритма Хаффмана.\
 __Входные данные:__ функций байтовая строка и вероятностная модель источника сообщения. \
__Выходные данные:__ функций байтовая строка.\
Какие метаданные необходимо записать для правильного декодирования сжатого сообщения? Как это может повлиять на итоговый коэффициент сжатия?
Напишите дополнительные функции для записи и чтения сжатого сообщения из файла.
*   Попробовать реализовать алгоритм арифметического кодирования и опытным путем установить при каких длинах строки левая и правая граница полуинтервала начинают совпадать при хранении их в типе double.\
__Входные данные:__ байтовая строка и вероятностная модель источника сообщения.\
__Выходные данные:__ число в формате представления double.

####  2. Преобразование Барроуза-Уиллера
* Написать алгоритм прямого BWT c прямым построением матрицы Барроуза-Уиллера.\
 __Входные данные:__ байтовая строка. \
__Выходные данные:__ байтовая строка.\
Написать алгоритм обратного BWT с прямым построением матрицы Барроуза-Уиллера. Проверить корректность работы программы на строке “0x62 0x61 0x6e 0x61 0x6e 0x61”. \
Оценить пространственную и временную сложность написанных алгоритмов.
* Написать алгоритм обратного BWT с использованием порождаемой сдвигом и сортировкой перестановки.  Сортировку последнего столбца осуществить с помощью сортировки подсчетом.\
Оценить пространственную и временную сложность.
*  Последовательно применить BWT и RLE к тестовым файлам и рассчитать коэффициент сжатия. Перед этим подумать о работоспособности алгоритма прямого BWT при больших размерах файла. Дополнить функцию разбиением исходной байтовой строки на блоки. Не забыть добавить в функцию обратного преобразования блочную обработку. Удостовериться, что модификация успешно работает на enwik7 и тексте на русском языке. Проанилизировать результат.

In [5]:
from collections import Counter
import numpy as np

def calculate_entropy(data: bytes, ms: int = 1) -> float:
        """Расчет энтропии для символов в ms байт"""
        if not data:
            return 0.0

        n = len(data)
        if n % ms != 0:
            # дополняем до кратности ms
            data += b'\x00' * (ms - (n % ms))
            n = len(data)

        symbols = [data[i : i + ms] for i in range(0, n, ms)]
        counts = Counter(symbols)

        return sum(( -( p := count / len(symbols)) * np.log(p) for count in counts.values()))

In [6]:
def analyze_entropy_dependency(text: str) -> None:
    print("\nИсследование энтропии от Ms (длины кода):")
    clean_data = bytes([b for b in text.encode('ascii', 'ignore') if b < 128])
    for ms in range(1, 5):
        ent = calculate_entropy(clean_data, ms)
        print(f"Ms = {ms} байт: {ent:.4f} бит/символ (всего {ent*len(clean_data)/ms/8:.2f} байт)")

In [7]:
with open("resources/dc2-prologue.txt", "r", encoding="utf-8") as f:
    analyze_entropy_dependency(f.read())


Исследование энтропии от Ms (длины кода):
Ms = 1 байт: 3.0961 бит/символ (всего 4876.81 байт)
Ms = 2 байт: 5.4265 бит/символ (всего 4273.72 байт)
Ms = 3 байт: 6.8588 бит/символ (всего 3601.16 байт)
Ms = 4 байт: 7.4114 бит/символ (всего 2918.46 байт)


In [ ]:
class MoveToFrontTransform:
    """Преобразование Move-To-Front"""

    @staticmethod
    def encode(data: bytes) -> bytes:
        alphabet = list(range(256))
        result = bytearray()
        for byte in data:
            idx = alphabet.index(byte)
            result.append(idx)
            alphabet.insert(0, alphabet.pop(idx))
        return bytes(result)

    @staticmethod
    def decode(data: bytes) -> bytes:
        alphabet = list(range(256))
        result = bytearray()
        for idx in data:
            byte = alphabet[idx]
            result.append(byte)
            alphabet.insert(0, alphabet.pop(idx))
        return bytes(result)

mtf = MoveToFrontTransform

## __Задание 3__

####  1. Преобразование Барроуза-Уиллера
* Написать функцию преобразования суффиксного массива в последний столбец матрицы Барроуза-Уиллера. \
__Входные данные__: массив из чисел типа int32 \
__Выходные данные__: байтовая строка.

#### 2. Методы Лемпеля-Зива
* Написать функцию кодирования и декодирования алгоритмом LZ77 \
 __Входные данные__: байтовая строка и размер буфера \
__Выходные данные__: байтовая строка. \
Какие метаданные необходимо записать для правильного декодирования сжатого сообщения?
* Адаптировать функции кодирования и декодирования для реализации алгоритма LZSS
* Написать функцию кодирования и декодирования алгоритмом LZ78 \
 __Входные данные__: байтовая строка \
__Выходные данные__: байтовая строка. \
Какие метаданные необходимо записать для правильного декодирования сжатого сообщения? \
Измените функцию кодирования и декодирования ограничив максимальный размер словаря.
* Адаптировать функции кодирования и декодирования для реализации алгоритма LZW. \
Подумать необходимо ли для декодирования сохранять начальный словарь.
* Исследовать как меняется коэффициент сжатия и время работы программы в зависимости от размера буфера и размера словаря для алгоритмов LZSS и LZW.


## __Задание 4__

#### 1. Преобразование Барроуза-Уиллера.
* Реализовать эффективное вычисление прямого преобразование Барроуза-Уиллера на основе алгоритма нахождения суффиксного массива. Нахождение суффиксного массива можно реализовать самостоятельно/воспользоваться библиотечной функцией/воспользоваться готовой реализацией. Узнать оценку временной и пространственной асимптотики выбранной реализации.
#### 2. Канонические коды Хаффмана
* Реализовать функцию построение набора канонических кодов Хаффмана по их длинам. Изменить функции кодирования и декодирования алгоритмом Хаффмана, использовав канонические коды и оптимизировав размер метаданных (словаря).


## __Отчет__

#### 1. Содержание отчета
* Теоретическая часть, в которой кратко описываются реализованные алгоритмы, их временная и пространственная сложность;
* Практическая часть, в которой приводятся результаты работы различных компрессоров, а также выбранные при их реализации параметры и исследуется их эффективность;
* Приложения с кодом и ссылкой на репозиторий гитхаба. Вместе с кодом в репозитории должны находиться исходные тестовые файлы, их кодированная и декодированная версии.

#### 2. Задание на лабораторную работу
*  Исследовать зависимость энтропии сообщениия от размера блоков, на которые разбивается текст, подаваемый на вход BWT+MTF, для тестовых файлов. Вывести полученные результаты на график, сравнить с исходной энтропией и сделать вывод об оптимальном размере блока.
* Исследовать зависимость коэффициента сжатия от размера буфера для алгоритма LZSS. Вывести полученные результаты на график и сделать вывод об оптимальном размере буфера.
* Исследовать зависимость коэффициента сжатия от размера словаря для алгоритма LZW. Вывести полученные результаты на график и сделать вывод об оптимальном размере размере словаря
* Собрать на основе реализованных алгоритмов следующие компрессоры:
  1. HA
  2. Run-length encoding (RLE)
  3. BWT + RLE
  4. BWT + MTF + HA
  5. BWT + MTF + RLE + HA
  6. LZSS
  7. LZSS + HA
  8. LZW
  9. LZW + HA

  Выбрать размер блока для BWT на основе проведенных исследований. \
  Для алгоритмов семейства LZ выбрать размер буфера/словаря на основе проведенных исследований.
* Удостовериться в корректной компрессии и декомпрессии данных для всех компрессоров.
* Исследовать эффективность компрессоров для всех тестовых данных. Свести размеры исходных файлов, файлов после компрессии и файлов после декомпресси в байтах и соответствующие коэффициенты сжатия в таблицу.

#### __Комментарии к оформлению отчета (может дополняться)__
1. Все графики должны быть корректно оформлены, должны присутствовать подписи к осям и легенда.
2. Таблицы в отчете должны быть читаемыми. При необходимости можно разбивать таблицы на несколько частей.